In [1]:
!pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"

print("✅ Dependencies ready!")

✅ Dependencies ready!


In [2]:
from langchain_core.tools import tool
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import render_text_description
from langchain_core.runnables import RunnablePassthrough
from typing import Any, Dict, Optional, TypedDict
from langchain_community.llms import Ollama
import json
import os

print("✅ All imports successful!")

✅ All imports successful!


In [3]:
# Initialize Ollama with Phi-3
model = Ollama(model="phi3")

print("✅ Model initialized!")
print(f"✅ Model type: {type(model)}")

✅ Model initialized!
✅ Model type: <class 'langchain_community.llms.ollama.Ollama'>


In [4]:
# Define Tools
from langchain_core.tools import tool

@tool
def multiply(x: float, y: float) -> float:
    """Multiply two numbers together."""
    return x * y

@tool
def add(x: int, y: int) -> int:
    """Add two numbers."""
    return x + y

tools = [multiply, add]

# inspect the tools
for t in tools:
    print("--")
    print(f"Name: {t.name}")
    print(f"Description: {t.description}")
    print(f"Arguments: {t.args}")
    print()

# Test
print("Testing multiply(4, 5):", multiply.invoke({"x": 4, "y": 5}))
print("Testing add(3, 7):", add.invoke({"x": 3, "y": 7}))

--
Name: multiply
Description: Multiply two numbers together.
Arguments: {'x': {'title': 'X', 'type': 'number'}, 'y': {'title': 'Y', 'type': 'number'}}

--
Name: add
Description: Add two numbers.
Arguments: {'x': {'title': 'X', 'type': 'integer'}, 'y': {'title': 'Y', 'type': 'integer'}}

Testing multiply(4, 5): 20.0
Testing add(3, 7): 10


In [7]:
# Render tool descriptions
from langchain_core.tools import render_text_description

rendered_tools = render_text_description(tools)
print("TOOL DESCRIPTIONS")
print(rendered_tools)

TOOL DESCRIPTIONS
multiply(x: float, y: float) -> float - Multiply two numbers together.
add(x: int, y: int) -> int - Add two numbers.


In [9]:
system_prompt = f"""\
You are an assistant that has access to the following set of tools. 
Here are the names and descriptions for each tool:

{rendered_tools}

Given the user input, return the name and input of the tool to use. 
Return your response as a JSON blob with 'name' and 'arguments' keys.

The `arguments` should be a dictionary, with keys corresponding 
to the argument names and the values corresponding to the requested values.
"""

print("📋 SYSTEM PROMPT")
print(system_prompt)

📋 SYSTEM PROMPT
You are an assistant that has access to the following set of tools. 
Here are the names and descriptions for each tool:

multiply(x: float, y: float) -> float - Multiply two numbers together.
add(x: int, y: int) -> int - Add two numbers.

Given the user input, return the name and input of the tool to use. 
Return your response as a JSON blob with 'name' and 'arguments' keys.

The `arguments` should be a dictionary, with keys corresponding 
to the argument names and the values corresponding to the requested values.



In [11]:
# full prompt template
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("user", "{input}")]
)

In [13]:
# Test raw output
from langchain_community.llms import Ollama

model = Ollama(model="phi3")
chain = prompt | model

# Test with a question
message = chain.invoke({"input": "what's 3 plus 1132"})

print("🔍 RAW MODEL OUTPUT")
print(message)
print("📝 OUTPUT TYPE:", type(message))

🔍 RAW MODEL OUTPUT
```json
{
  "name": "add",
  "arguments": {
    "x": 3,
    "y": 1132
  }
}
```
📝 OUTPUT TYPE: <class 'str'>


In [ ]:

# Create and test the chain with parser
chain_with_parser = prompt | model | JsonOutputParser()
result = chain_with_parser.invoke({"input": "what's 3 plus 1132"})

print("🔍 PARSED OUTPUT")
print(f"Type: {type(result)}")
print(f"Content: {result}")
print(f"\nName: {result['name']}")
print(f"Arguments: {result['arguments']}")

🔍 PARSED OUTPUT
Type: <class 'dict'>
Content: {'name': 'add', 'arguments': {'x': 3, 'y': 1132}}

Name: add
Arguments: {'x': 3, 'y': 1132}


In [19]:
# Create the invoke_tool function
from typing import Any, Dict, Optional, TypedDict
from langchain_core.runnables import RunnableConfig

class ToolCallRequest(TypedDict):
    """A typed dict that shows the inputs into the invoke_tool function."""
    name: str
    arguments: Dict[str, Any]

def invoke_tool(
    tool_call_request: ToolCallRequest, 
    config: Optional[RunnableConfig] = None
):
    """A function that we can use to perform a tool invocation.
    
    Args:
        tool_call_request: a dict that contains the keys name and arguments.
            The name must match the name of a tool that exists.
            The arguments are the arguments to that tool.
        config: Configuration information for LangChain.
    
    Returns:
        output from the requested tool
    """
    # Create a lookup dictionary for tools
    tool_name_to_tool = {tool.name: tool for tool in tools}
    
    # Get the requested tool
    name = tool_call_request["name"]
    requested_tool = tool_name_to_tool[name]
    
    # Invoke the tool with the arguments
    return requested_tool.invoke(tool_call_request["arguments"], config=config)

test_result = invoke_tool({"name": "multiply", "arguments": {"x": 3, "y": 5}})
print(f"Testing multiply(3, 5): {test_result}")

test_result2 = invoke_tool({"name": "add", "arguments": {"x": 10, "y": 20}})
print(f"Testing add(10, 20): {test_result2}")

Testing multiply(3, 5): 15.0
Testing add(10, 20): 30


In [21]:
# Create the complete chain

# Method 1: Simple chain (returns just the tool output)
chain = prompt | model | JsonOutputParser() | invoke_tool

print("🧪 TESTING THE COMPLETE CHAIN")

# Test 1: Addition
result1 = chain.invoke({"input": "what's 3 plus 1132"})
print(f"\n📝 Question: what's 3 plus 1132")
print(f"✅ Answer: {result1}")

# Test 2: Multiplication
result2 = chain.invoke({"input": "what's thirteen times 4.14137281"})
print(f"\n📝 Question: what's thirteen times 4.14137281")
print(f"✅ Answer: {result2}")

# Test 3: Another addition
result3 = chain.invoke({"input": "add 45 and 67"})
print(f"\n📝 Question: add 45 and 67")
print(f"✅ Answer: {result3}")

# Test 4: Another multiplication
result4 = chain.invoke({"input": "multiply 12 by 8"})
print(f"\n📝 Question: multiply 12 by 8")
print(f"✅ Answer: {result4}")

🧪 TESTING THE COMPLETE CHAIN

📝 Question: what's 3 plus 1132
✅ Answer: 1135

📝 Question: what's thirteen times 4.14137281
✅ Answer: 53.83784653

📝 Question: add 45 and 67
✅ Answer: 112

📝 Question: multiply 12 by 8
✅ Answer: 96.0


In [22]:
# Chain that returns both inputs and outputs
chain_with_details = (
    prompt 
    | model 
    | JsonOutputParser() 
    | RunnablePassthrough.assign(output=invoke_tool)
)

print("✅ Enhanced chain created!")
print("This chain returns: tool_name, arguments, AND output")

# Test
result = chain_with_details.invoke({"input": "what's 3 plus 1132"})

print("🔍 ENHANCED OUTPUT")
print(f"Tool called: {result['name']}")
print(f"Arguments: {result['arguments']}")
print(f"Output: {result['output']}")

✅ Enhanced chain created!
This chain returns: tool_name, arguments, AND output
🔍 ENHANCED OUTPUT
Tool called: add
Arguments: {'x': 3, 'y': 1132}
Output: 1135


In [23]:
# Test various question formats
print("📊 TESTING DIFFERENT QUESTION FORMATS")

test_questions = [
    "what's 3 plus 1132",
    "what's thirteen times 4.14137281",
    "add 45 and 67",
    "multiply 12 by 8",
    "can you add 100 and 200 for me?",
    "I need to multiply 5 and 6",
]

for question in test_questions:
    print(f"\n❓ Question: {question}")
    try:
        result = chain.invoke({"input": question})
        print(f"✅ Answer: {result}")
    except Exception as e:
        print(f"❌ Error: {e}")

📊 TESTING DIFFERENT QUESTION FORMATS

❓ Question: what's 3 plus 1132
✅ Answer: 1135

❓ Question: what's thirteen times 4.14137281
✅ Answer: 53.83784653

❓ Question: add 45 and 67
✅ Answer: 112

❓ Question: multiply 12 by 8
✅ Answer: 96.0

❓ Question: can you add 100 and 200 for me?
✅ Answer: 300

❓ Question: I need to multiply 5 and 6
✅ Answer: 30.0


In [25]:
# Custom Tools

# TOOL 1: Temperature Converter
@tool
def celsius_to_fahrenheit(celsius: float) -> float:
    """Convert Celsius to Fahrenheit."""
    return (celsius * 9/5) + 32

# TOOL 2: String Reverser
@tool
def reverse_string(text: str) -> str:
    """Reverse a string."""
    return text[::-1]

# TOOL 3: Word Counter
@tool
def count_words(text: str) -> int:
    """Count the number of words in a text."""
    return len(text.split())


tools = [celsius_to_fahrenheit, reverse_string, count_words]
rendered_tools = render_text_description(tools)


In [26]:
# Updated system prompt with custom tools
system_prompt = f"""\
You are an assistant that has access to the following set of tools. 
Here are the names and descriptions for each tool:

{rendered_tools}

Given the user input, return the name and input of the tool to use. 
Return your response as a JSON blob with 'name' and 'arguments' keys.

The `arguments` should be a dictionary, with keys corresponding 
to the argument names and the values corresponding to the requested values.

Use the tools when appropriate. If the user asks something that doesn't match
a tool, respond with a message explaining what you can do.
"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("user", "{input}")]
)


In [27]:
# Cell: Complete chain with custom tools

parser = JsonOutputParser()

class ToolCallRequest(TypedDict):
    name: str
    arguments: Dict[str, Any]

def invoke_tool(
    tool_call_request: ToolCallRequest, 
    config: Optional[RunnableConfig] = None
):
    tool_name_to_tool = {tool.name: tool for tool in tools}
    name = tool_call_request["name"]
    requested_tool = tool_name_to_tool[name]
    return requested_tool.invoke(tool_call_request["arguments"], config=config)

chain = prompt | model | parser | invoke_tool
chain_with_details = (
    prompt 
    | model 
    | parser 
    | RunnablePassthrough.assign(output=invoke_tool)
)


In [30]:
test_questions = [
    "What is 25°C in Fahrenheit?",
    "Reverse the word 'hello'",
    "Count how many words are in 'this is a test sentence'",
    "What is 100°C in Fahrenheit?",
    "Reverse 'LangChain is awesome'",
    "How many words in 'one two three four five'?"
]

for question in test_questions:
    print(f"\n{'─'*70}")
    print(f"Question: {question}")
    print(f"{'─'*70}")
    try:
        result = chain.invoke({"input": question})
        print(f"✅ Answer: {result}")
    except Exception as e:
        print(f"❌ Error: {e}")


──────────────────────────────────────────────────────────────────────
Question: What is 25°C in Fahrenheit?
──────────────────────────────────────────────────────────────────────
✅ Answer: 77.0

──────────────────────────────────────────────────────────────────────
Question: Reverse the word 'hello'
──────────────────────────────────────────────────────────────────────
✅ Answer: olleh

──────────────────────────────────────────────────────────────────────
Question: Count how many words are in 'this is a test sentence'
──────────────────────────────────────────────────────────────────────
✅ Answer: 5

──────────────────────────────────────────────────────────────────────
Question: What is 100°C in Fahrenheit?
──────────────────────────────────────────────────────────────────────
✅ Answer: 212.0

──────────────────────────────────────────────────────────────────────
Question: Reverse 'LangChain is awesome'
──────────────────────────────────────────────────────────────────────
✅ Answer: